<a href="https://colab.research.google.com/github/sridhartroy/AIML/blob/main/SE_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyarrow pandas scikit-learn joblib xgboost

In [2]:
from google.colab import files
uploaded = files.upload()
parquet_name = next(iter(uploaded.keys()))
parquet_name

Saving yellow_tripdata_2025-10.parquet to yellow_tripdata_2025-10.parquet


'yellow_tripdata_2025-10.parquet'

In [3]:
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


# ---------- Helpers ----------

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def make_preprocessor(numeric_cols, cat_cols):

    # XGBoost handles sparse outputs well; keep OneHotEncoder sparse.

    return ColumnTransformer(

        transformers=[

            ("num", "passthrough", numeric_cols),

            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),

        ]

    )

def xgb_reg(seed, n_estimators=600, learning_rate=0.05, max_depth=8,

            subsample=0.85, colsample_bytree=0.85):

    return XGBRegressor(

        n_estimators=n_estimators,

        learning_rate=learning_rate,

        max_depth=max_depth,

        subsample=subsample,

        colsample_bytree=colsample_bytree,

        reg_lambda=1.0,

        objective="reg:squarederror",

        n_jobs=-1,

        random_state=seed,

    )

def load_and_clean(parquet_path):

    df = pd.read_parquet(parquet_path)

    df.columns = [c.strip() for c in df.columns]

    required = [

        "tpep_pickup_datetime","tpep_dropoff_datetime",

        "PULocationID","DOLocationID",

        "trip_distance",

        "fare_amount","tip_amount","tolls_amount",

        "improvement_surcharge","congestion_surcharge",

        "total_amount",

    ]

    missing = [c for c in required if c not in df.columns]

    if missing:

        raise ValueError(f"Missing columns: {missing}\nFound: {list(df.columns)}")

    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")

    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    df["trip_duration_sec"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds()

    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

    df["pickup_dow"] = df["tpep_pickup_datetime"].dt.dayofweek

    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

    df = df.dropna(subset=[

        "trip_duration_sec","trip_distance","fare_amount","total_amount",

        "PULocationID","DOLocationID","pickup_hour","pickup_dow","pickup_month",

        "tip_amount","tolls_amount","improvement_surcharge","congestion_surcharge"

    ]).copy()

    # filters

    df = df[

        (df["trip_duration_sec"] >= 60) & (df["trip_duration_sec"] <= 3*60*60) &

        (df["trip_distance"] >= 0.1) & (df["trip_distance"] <= 60) &

        (df["fare_amount"] >= 2) & (df["fare_amount"] <= 250) &

        (df["total_amount"] >= 3) & (df["total_amount"] <= 400)

    ].copy()

    # categorical zones

    df["PULocationID"] = df["PULocationID"].astype(int).astype(str)

    df["DOLocationID"] = df["DOLocationID"].astype(int).astype(str)

    # numeric types

    for c in ["trip_distance","trip_duration_sec","fare_amount","tip_amount","tolls_amount",

              "improvement_surcharge","congestion_surcharge","total_amount",

              "pickup_hour","pickup_dow","pickup_month"]:

        df[c] = df[c].astype(float)

    df["total_no_tip"] = (df["total_amount"] - df["tip_amount"]).clip(lower=0.0)

    return df


# ---------- Config ----------

SEED = 42

SAMPLE_ROWS = 500_000         # change if you want

KFOLDS = 5

PREDICT_TOLLS = True          # set False if you want faster

# ---------- Load data ----------

df = load_and_clean(parquet_name)

print("Clean rows available:", f"{len(df):,}")

if len(df) > SAMPLE_ROWS:

    df = df.sample(n=SAMPLE_ROWS, random_state=SEED).reset_index(drop=True)

print("Rows used:", f"{len(df):,}")

# Base quote-time features

base_num = ["trip_distance", "pickup_hour", "pickup_dow", "pickup_month"]

base_cat = ["PULocationID", "DOLocationID"]

X_base = df[base_num + base_cat]

# Train/test split for honest final metrics

X_tr, X_te, df_tr, df_te = train_test_split(X_base, df, test_size=0.2, random_state=SEED)

# ---------- Stage 1: ETA model ----------

pre_eta = make_preprocessor(base_num, base_cat)

eta_reg = xgb_reg(SEED, n_estimators=500, learning_rate=0.06, max_depth=8)

# OOF ETA for train rows

kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)

eta_oof = np.zeros(len(df_tr), dtype=float)

X_tr_df = df_tr[base_num + base_cat].reset_index(drop=True)

y_eta_tr = df_tr["trip_duration_sec"].reset_index(drop=True)

for tr_idx, va_idx in kf.split(X_tr_df):

    pipe_k = Pipeline([("pre", pre_eta), ("model", eta_reg)])

    pipe_k.fit(X_tr_df.loc[tr_idx], y_eta_tr.loc[tr_idx])

    eta_oof[va_idx] = pipe_k.predict(X_tr_df.loc[va_idx])

# Fit final ETA model on full training

eta_pipe = Pipeline([("pre", pre_eta), ("model", eta_reg)])

eta_pipe.fit(X_tr_df, y_eta_tr)

eta_pred_test = eta_pipe.predict(df_te[base_num + base_cat])

eta_mae_sec = float(mean_absolute_error(df_te["trip_duration_sec"], eta_pred_test))

print("ETA MAE (sec):", round(eta_mae_sec, 1), "| minutes:", round(eta_mae_sec/60, 2))

# ---------- Stage 2: Fare model (uses eta_pred_sec) ----------

df_tr2 = df_tr.reset_index(drop=True).copy()

df_tr2["eta_pred_sec"] = eta_oof

df_te2 = df_te.copy()

df_te2["eta_pred_sec"] = eta_pred_test

fare_num = base_num + ["eta_pred_sec"]

fare_cat = base_cat

pre_fare = make_preprocessor(fare_num, fare_cat)

fare_reg = xgb_reg(SEED, n_estimators=700, learning_rate=0.05, max_depth=8)

fare_pipe = Pipeline([("pre", pre_fare), ("model", fare_reg)])

fare_pipe.fit(df_tr2[fare_num + fare_cat], df_tr2["fare_amount"])

fare_pred_test = fare_pipe.predict(df_te2[fare_num + fare_cat])

fare_mae = float(mean_absolute_error(df_te2["fare_amount"], fare_pred_test))

fare_rmse = rmse(df_te2["fare_amount"], fare_pred_test)

print("Fare MAE ($):", round(fare_mae, 2), "| RMSE ($):", round(fare_rmse, 2))

# ---------- Optional: Tolls model ----------

tolls_pipe = None

tolls_pred_test = np.zeros(len(df_te2), dtype=float)

tolls_mae = tolls_rmse = None

if PREDICT_TOLLS:

    tolls_reg = xgb_reg(SEED, n_estimators=500, learning_rate=0.06, max_depth=7)

    tolls_pipe = Pipeline([("pre", pre_fare), ("model", tolls_reg)])

    tolls_pipe.fit(df_tr2[fare_num + fare_cat], df_tr2["tolls_amount"])

    tolls_pred_test = tolls_pipe.predict(df_te2[fare_num + fare_cat])

    tolls_mae = float(mean_absolute_error(df_te2["tolls_amount"], tolls_pred_test))

    tolls_rmse = rmse(df_te2["tolls_amount"], tolls_pred_test)

    print("Tolls MAE ($):", round(tolls_mae, 2), "| RMSE ($):", round(tolls_rmse, 2))

# ---------- Total estimate (no tip) ----------

total_est_no_tip = (

    fare_pred_test

    + df_te2["congestion_surcharge"].to_numpy()

    + df_te2["improvement_surcharge"].to_numpy()

    + tolls_pred_test

)

y_total_no_tip = df_te2["total_no_tip"].to_numpy()

total_no_tip_mae = float(mean_absolute_error(y_total_no_tip, total_est_no_tip))

total_no_tip_rmse = rmse(y_total_no_tip, total_est_no_tip)

print("Total(no tip) MAE ($):", round(total_no_tip_mae, 2), "| RMSE ($):", round(total_no_tip_rmse, 2))

# ---------- Save artifacts ----------

joblib.dump(eta_pipe, "eta_model_xgb.pkl")

joblib.dump(fare_pipe, "fare_model_xgb.pkl")

if PREDICT_TOLLS and tolls_pipe is not None:

    joblib.dump(tolls_pipe, "tolls_model_xgb.pkl")

metrics = {

    "rows_used": int(len(df)),

    "sample_rows": SAMPLE_ROWS,

    "kf": KFOLDS,

    "eta_mae_seconds": eta_mae_sec,

    "fare_mae_usd": fare_mae,

    "fare_rmse_usd": fare_rmse,

    "tolls_enabled": PREDICT_TOLLS,

    "tolls_mae_usd": tolls_mae,

    "tolls_rmse_usd": tolls_rmse,

    "total_no_tip_mae_usd": total_no_tip_mae,

    "total_no_tip_rmse_usd": total_no_tip_rmse,

    "notes": [

        "Two-stage model: ETA (trip_duration_sec) -> Fare (fare_amount) with OOF ETA feature.",

        "Total(no tip) computed as fare_pred + congestion + improvement + toll_pred.",

        "Tip excluded (quote-time realism)."

    ]

}

with open("metrics_xgb.json", "w") as f:

    json.dump(metrics, f, indent=2)

print("\nSaved: eta_model_xgb.pkl, fare_model_xgb.pkl, metrics_xgb.json",

      ("+ tolls_model_xgb.pkl" if PREDICT_TOLLS else ""))


Clean rows available: 3,243,096
Rows used: 500,000
ETA MAE (sec): 241.8 | minutes: 4.03
Fare MAE ($): 2.44 | RMSE ($): 4.76
Tolls MAE ($): 0.35 | RMSE ($): 1.26
Total(no tip) MAE ($): 3.23 | RMSE ($): 5.74

Saved: eta_model_xgb.pkl, fare_model_xgb.pkl, metrics_xgb.json + tolls_model_xgb.pkl
